In [ ]:
import torch
import torch.nn.functional as F

from onomatron.data import Vocab, load_names, make_ngram_pairs, split_names
from onomatron.linear import LinearBigram
from onomatron.ngram import NGramModel
from onomatron.utils import lr_range_test, plot_history, plot_lr_range, train

%matplotlib inline

# 3 — Simple neural net: the linear bigram

Goal: match the count-based bigram score with a model trained by gradient
descent. 

One linear layer over one-hot contexts,
`logits = xenc @ W`, trained with cross-entropy on the next character.


## Load names, split train/val/test, build the vocab

Same fixed seed and ratios as the other notebooks, so every model is compared
on identical splits. Vocab from train only: no leakage from val/test.

In [ ]:
names = load_names("../data/processed/names.txt")
train_names, val_names, test_names = split_names(names, seed=42)
vocab = Vocab(train_names)  # vocab from train only: no leakage from val/test
print(
    f"{len(names)} names -> {len(train_names)} train / {len(val_names)} val"
    f" / {len(test_names)} test, vocab of {len(vocab)} chars"
)

## Bigrams -> tensors, per split

In [ ]:
def bigrams(name_list: list[str]) -> tuple[torch.Tensor, torch.Tensor]:
    pairs = make_ngram_pairs(name_list, vocab, n=2)  # (N, 2): context char, target char
    xs, ys = pairs[:, 0], pairs[:, 1]
    return F.one_hot(xs, num_classes=len(vocab)).float(), ys


xenc, ys = bigrams(train_names)
xenc_val, ys_val = bigrams(val_names)
xenc_test, ys_test = bigrams(test_names)
print(
    f"train {xenc.shape[0]} bigrams, xenc {tuple(xenc.shape)}"
    f" | val {xenc_val.shape[0]} | test {xenc_test.shape[0]}"
)

## LR range test

One fresh model per learning rate (fixed seed), 100 SGD steps each, then plot
the final loss on a log scale. We keep a decade below the minimum.


In [ ]:
lrs = (10 ** torch.linspace(-3, 3, 100)).tolist()  # 1e-3 -> 1e3, log scale
losses = lr_range_test(lambda seed: LinearBigram(vocab, seed=seed), xenc, ys, lrs)

best_lr = plot_lr_range(lrs, losses)
lr = best_lr / 10  # one decade below the minimum, safety margin
print(f"minimum at lr={best_lr:.4f} -> using lr={lr:.4f}")

## Train on train, watch val

In [ ]:
model = LinearBigram(vocab, seed=42)
opt = torch.optim.SGD(model.parameters(), lr=lr)

history = train(
    model,
    xenc,
    ys,
    opt,
    steps=5000,
    x_val=xenc_val,
    y_val=ys_val,
    eval_every=250,
    seed=42,
)
plot_history(history)
print(
    f"final train loss {history.train_losses[-1]:.4f}"
    f" | val NLL {history.val[-1][1]:.4f}"
)

bigram = NGramModel.fit(train_names, vocab, n=2)
trigram = NGramModel.fit(train_names, vocab, n=3)
print(f"NLL linear NN : {model.nll(xenc, ys):.4f}")
print(f"NLL bigram    : {bigram.nll(train_names):.4f}")
print(f"NLL trigram   : {trigram.nll(train_names):.4f}")

## Sample with single Layer NN (min 3 chars)

In [ ]:
model.sample(n=20, seed=42)

## Final evaluation on test

Evaluated once, after training and lr selection on train/val.

In [ ]:
print(
    f"linear NN: train NLL {model.nll(xenc, ys):.4f}"
    f" | val NLL {model.nll(xenc_val, ys_val):.4f}"
    f" | test NLL {model.nll(xenc_test, ys_test):.4f}"
)